<a href="https://colab.research.google.com/github/endrissuofe/cost-leakage-analysis/blob/main/Duplicate_Detection_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# 1. Setup basic parameters for our synthetic data
np.random.seed(42) # Ensures we get the same random data every time
num_records = 10000
agents = [f'AGT-{str(i).zfill(3)}' for i in range(1, 51)] # 50 unique agents
standard_payment = 15.50 # Flat fee paid to agents per verification

# 2. Generate the base authentic records
data = {
    'Verification_ID': [f'VER-{str(i).zfill(6)}' for i in range(1, num_records + 1)],
    'Agent_ID': np.random.choice(agents, num_records),
    'Property_ID': np.random.randint(1000, 9000, num_records), # The location being verified
    'Submission_Date': [datetime(2025, 1, 1) + timedelta(days=random.randint(0, 30)) for _ in range(num_records)],
    'Payment_Amount': [standard_payment] * num_records
}

df_authentic = pd.DataFrame(data)

# 3. Inject Duplicates (Simulating the financial leakage)
# We will randomly select 500 rows and duplicate them to simulate fraudulent/accidental double-submissions
df_duplicates = df_authentic.sample(n=500, random_state=1)

# Combine the authentic data with the duplicates
df_messy = pd.concat([df_authentic, df_duplicates])

# Shuffle the dataset so the duplicates are hidden randomly throughout
df_messy = df_messy.sample(frac=1, random_state=42).reset_index(drop=True)

# 4. View the generated data and export it
print(f"Generated dataset with {len(df_messy)} rows (includes hidden duplicates).")
display(df_messy.head())

# Export the raw, messy dataset to start our analysis
df_messy.to_csv('raw_agent_verifications.csv', index=False)

Generated dataset with 10500 rows (includes hidden duplicates).


,Verification_ID,Agent_ID,Property_ID,Submission_Date,Payment_Amount
0,VER-005119,AGT-036,1461,2025-01-13,15.5
1,VER-008932,AGT-010,8689,2025-01-25,15.5
2,VER-008516,AGT-045,7980,2025-01-22,15.5
3,VER-007283,AGT-037,8762,2025-01-03,15.5
4,VER-007624,AGT-043,7193,2025-01-09,15.5


In [3]:
import sqlite3

# 1. Create an in-memory SQLite database
conn = sqlite3.connect(':memory:')

# 2. Load our messy dataframe into a SQL table named 'agent_verifications'
df_messy.to_sql('agent_verifications', conn, index=False)

# 3. Create a helper function to run our SQL queries and display the results cleanly
def run_query(query):
    return pd.read_sql_query(query, conn)

print("SQL Environment setup complete. The 'agent_verifications' table is ready to query.")

SQL Environment setup complete. The 'agent_verifications' table is ready to query.


In [4]:
# SQL Query 1: Identify duplicate verification submissions
query_duplicates = """
SELECT
    Verification_ID,
    COUNT(*) as Submission_Count
FROM
    agent_verifications
GROUP BY
    Verification_ID
HAVING
    COUNT(*) > 1
ORDER BY
    Submission_Count DESC
LIMIT 10;
"""

print("--- Sample of Duplicate Verification IDs ---")
display(run_query(query_duplicates))

--- Sample of Duplicate Verification IDs ---


,Verification_ID,Submission_Count
0,VER-009985,2
1,VER-009978,2
2,VER-009954,2
3,VER-009940,2
4,VER-009929,2
5,VER-009888,2
6,VER-009884,2
7,VER-009858,2
8,VER-009822,2
9,VER-009819,2


In [5]:
# SQL Query 2: Quantify the total estimated revenue leakage
query_leakage = """
WITH DuplicateCounts AS (
    SELECT
        Verification_ID,
        Payment_Amount,
        COUNT(*) as Submission_Count
    FROM
        agent_verifications
    GROUP BY
        Verification_ID,
        Payment_Amount
    HAVING
        COUNT(*) > 1
)
SELECT
    SUM((Submission_Count - 1) * Payment_Amount) AS Total_Financial_Leakage,
    SUM(Submission_Count - 1) AS Total_Duplicate_Jobs
FROM
    DuplicateCounts;
"""

print("\n--- Total Cost Leakage Analysis ---")
display(run_query(query_leakage))


--- Total Cost Leakage Analysis ---


,Total_Financial_Leakage,Total_Duplicate_Jobs
0,7750.0,500


In [6]:
# SQL Query 3: Find which agents are driving the highest leakage
query_agent_leakage = """
WITH DuplicateSubmissions AS (
    SELECT
        Agent_ID,
        Verification_ID,
        Payment_Amount,
        (COUNT(*) - 1) AS Invalid_Submissions
    FROM
        agent_verifications
    GROUP BY
        Agent_ID, Verification_ID, Payment_Amount
    HAVING
        COUNT(*) > 1
)
SELECT
    Agent_ID,
    SUM(Invalid_Submissions) AS Total_Duplicate_Submissions,
    SUM(Invalid_Submissions * Payment_Amount) AS Financial_Leakage_Per_Agent
FROM
    DuplicateSubmissions
GROUP BY
    Agent_ID
ORDER BY
    Total_Duplicate_Submissions DESC
LIMIT 5;
"""

print("\n--- Top 5 Agents Driving Financial Leakage ---")
display(run_query(query_agent_leakage))


--- Top 5 Agents Driving Financial Leakage ---


,Agent_ID,Total_Duplicate_Submissions,Financial_Leakage_Per_Agent
0,AGT-035,18,279.0
1,AGT-037,17,263.5
2,AGT-047,16,248.0
3,AGT-007,16,248.0
4,AGT-042,15,232.5


In [8]:
import pandas as pd
import numpy as np
import sqlite3

# --- PART 1: GENERATE SYNTHETIC DATA (Simulating Financial Loss Scenario) ---
np.random.seed(42) # Ensures consistent random data
num_records = 10000
agents = [f'AGT-{str(i).zfill(3)}' for i in range(1, 51)] # 50 unique field agents
standard_payment = 15.50 # Standard payout per verification

# Generate base authentic verification records
base_data = pd.DataFrame({
    'Verification_ID': [f'VER-{str(i).zfill(6)}' for i in range(1, num_records + 1)],
    'Agent_ID': np.random.choice(agents, num_records),
    'Payment_Amount': [standard_payment] * num_records
})

# Inject 500 duplicates to simulate financial leakage from double-submissions
duplicates = base_data.sample(n=500, random_state=1)
df_messy = pd.concat([base_data, duplicates]).sample(frac=1, random_state=42).reset_index(drop=True)


# --- PART 2: SET UP SQL ENVIRONMENT ---
# Create an in-memory SQLite database and load our messy dataset
conn = sqlite3.connect(':memory:')
df_messy.to_sql('agent_verifications', conn, index=False)

def run_query(query):
    return pd.read_sql_query(query, conn)


# --- PART 3: EXECUTE SQL LOGIC ---

# Query 1: Quantify the total estimated revenue leakage
query_total_leakage = """
WITH DuplicateCounts AS (
    SELECT Verification_ID, Payment_Amount, COUNT(*) as Submission_Count
    FROM agent_verifications
    GROUP BY Verification_ID, Payment_Amount
    HAVING COUNT(*) > 1
)
SELECT
    SUM((Submission_Count - 1) * Payment_Amount) AS Total_Financial_Leakage,
    SUM(Submission_Count - 1) AS Total_Duplicate_Jobs
FROM DuplicateCounts;
"""
print("--- Overall Financial Leakage ---")
display(run_query(query_total_leakage))


# Query 2: Identify root cause by finding the top offending agents
query_agent_leakage = """
WITH DuplicateSubmissions AS (
    SELECT Agent_ID, Verification_ID, Payment_Amount, (COUNT(*) - 1) AS Invalid_Submissions
    FROM agent_verifications
    GROUP BY Agent_ID, Verification_ID, Payment_Amount
    HAVING COUNT(*) > 1
)
SELECT
    Agent_ID,
    SUM(Invalid_Submissions) AS Total_Duplicate_Submissions,
    SUM(Invalid_Submissions * Payment_Amount) AS Financial_Leakage
FROM DuplicateSubmissions
GROUP BY Agent_ID
ORDER BY Financial_Leakage DESC;
"""
print("\n--- Leakage Breakdown by Agent ---")
agent_leakage_df = run_query(query_agent_leakage)
display(agent_leakage_df.head(10))


# --- PART 4: EXPORT DATA FOR EXCEL SUMMARY ---
df_messy.to_csv('raw_agent_verifications.csv', index=False)
agent_leakage_df.to_csv('agent_leakage_summary.csv', index=False)
print("\nSQL Analysis complete. Summary data exported for Excel!")

--- Overall Financial Leakage ---


,Total_Financial_Leakage,Total_Duplicate_Jobs
0,7750.0,500



--- Leakage Breakdown by Agent ---


,Agent_ID,Total_Duplicate_Submissions,Financial_Leakage
0,AGT-035,18,279.0
1,AGT-037,17,263.5
2,AGT-047,16,248.0
3,AGT-007,16,248.0
4,AGT-042,15,232.5
5,AGT-028,15,232.5
6,AGT-045,14,217.0
7,AGT-032,14,217.0
8,AGT-008,13,201.5
9,AGT-046,12,186.0



SQL Analysis complete. Summary data exported for Excel!
